In [82]:
import pandas as pd
import requests
import time
import os
import logging
from scrapy import Selector
from scrapy.crawler import CrawlerProcess
import scrapy
import boto3
from dotenv import load_dotenv
import numpy as np
from fake_useragent import UserAgent
import plotly.express as px


load_dotenv()
AWS_KEY = os.environ["AWS_KEY"]
AWS_SECRET_KEY = os.environ["AWS_SECRET_KEY"]

In [83]:
##Data loading
#localisation of data: local or cloud
data_loc = "local" 
if data_loc == "cloud":

    data_path = {
        "gps" : "data_gps.csv",
        "weather" : "data_weather.csv",
        "booking" : "data_booking.csv"
    }
    session = boto3.Session(aws_access_key_id = AWS_KEY, 
                            aws_secret_access_key = AWS_SECRET_KEY, 
                            region_name = "eu-west-3")
    
    s3 = session.resource("s3")
else:
    data_booking = pd.read_csv("data_booking_26.csv", index_col=0)
    data_gps = pd.read_csv("data_gps.csv", index_col=0)
    data_weather = pd.read_csv("dl_S3_data_weather.csv", index_col=0)

In [84]:
data_booking["review_score"] = data_booking["review_score"].replace(',', '.', regex=True).astype(float)

In [85]:
data_booking

,city,name,url,review_score,description,location,lat,lon
0,Biarritz,Hôtel Barnea,https://www.booking.com/hotel/fr/hotelargieder...,8.9,1 grand lit double,missing value,NaN,NaN
1,Biarritz,Le Talaia Hôtel & Spa Biarritz,https://www.booking.com/hotel/fr/radisson-blu-...,8.8,1 lit simple,"{'tourism': 'Le Talaia Hôtel & Spa Biarritz', ...",43.478210,-1.564422
2,Biarritz,Hôtel Kemaris,https://www.booking.com/hotel/fr/best-western-...,8.4,1 grand lit double,"{'tourism': 'Hôtel Best Western Kemaris', 'roa...",43.476518,-1.562496
3,Biarritz,ALFRED HOTELS Les Halles,https://www.booking.com/hotel/fr/hotelanjou.fr...,9.1,1 grand lit double,missing value,NaN,NaN
4,Biarritz,Hôtel du Palais Biarritz,https://www.booking.com/hotel/fr/hoteldupalais...,9.5,2 lits simples,"{'tourism': 'Hôtel du Palais', 'road': ""Avenue...",43.486435,-1.556328
...,...,...,...,...,...,...,...,...
122,Saintes Maries de la mer,Hôtel Le Neptune en Camargue,https://www.booking.com/hotel/fr/le-neptune-en...,8.4,Hébergement géré par un particulier,missing value,NaN,NaN
123,Saintes Maries de la mer,Le Sauvage,https://www.booking.com/hotel/fr/le-sauvage-le...,9.3,Hébergement géré par un particulier,missing value,NaN,NaN
124,Saintes Maries de la mer,De l'eau et des oiseaux !,https://www.booking.com/hotel/fr/de-l-eau-et-d...,8.8,Hébergement géré par un particulier,missing value,NaN,NaN
125,Saintes Maries de la mer,Hôtel Spa Mas des Rièges,https://www.booking.com/hotel/fr/mas-des-riege...,9.0,"2 lits (1 canapé-lit, 1 grand lit double)",missing value,NaN,NaN


In [88]:
data_gps = data_gps.transpose().reset_index().rename(columns={"index":"city"})

In [89]:
data_weather

,city,date,cloudiness,temp_min,temp_max,humidity,pop,rain
0,Mont Saint Michel,2025-09-17,64,15.45,17.35,83,0.00,0.00
1,Mont Saint Michel,2025-09-17,36,18.65,20.72,63,0.00,0.00
2,Mont Saint Michel,2025-09-17,0,21.83,21.83,47,0.00,0.00
3,Mont Saint Michel,2025-09-17,0,18.60,18.60,72,0.00,0.00
4,Mont Saint Michel,2025-09-17,0,16.31,16.31,88,0.00,0.00
...,...,...,...,...,...,...,...,...
1395,La Rochelle,2025-09-21,92,16.31,16.31,70,0.92,1.70
1396,La Rochelle,2025-09-21,99,14.70,14.70,71,0.60,0.38
1397,La Rochelle,2025-09-22,98,14.78,14.78,70,0.30,0.00
1398,La Rochelle,2025-09-22,93,14.20,14.20,74,1.00,1.56


In [90]:
mean_weather = data_weather.drop("date", axis=1).groupby("city").mean().sort_values("temp_max", ascending=False)
mean_weather = mean_weather.reset_index().merge(data_gps, on="city", how="left", validate="1:1")
mean_weather["size"] = (mean_weather["temp_max"]-mean_weather["temp_max"].min())/(mean_weather["temp_max"].max()-mean_weather["temp_max"].min())
mean_weather.head()
#To be used for finer analysis
# data_weather.groupby(["city", "date"]).mean()

,city,cloudiness,temp_min,temp_max,humidity,pop,rain,lat,lon,size
0,Aix en Provence,59.225,22.82925,22.99850,52.175,0.07650,0.26875,43.529842,5.447474,1.000000
1,Marseille,56.225,22.60725,22.71225,65.200,0.08550,0.31575,43.296174,5.369953,0.959097
2,Collioure,56.900,21.82525,21.98825,69.100,0.08975,0.28325,42.525050,3.083155,0.855642
3,Avignon,57.000,21.80075,21.92025,59.350,0.15575,0.42200,43.949249,4.805901,0.845926
4,Cassis,59.725,21.65225,21.74025,64.725,0.07850,0.22650,43.214036,5.539632,0.820205


In [91]:
fig = px.scatter_map(
    mean_weather,
    lat="lat", lon="lon",
    color="temp_max",
    size="size", size_max=30,
    hover_name='city',
    hover_data={"temp_min":True, "temp_max":True},
    map_style="open-street-map",
    center={"lat":46.3, "lon":2.25},
    zoom=5, height=720,
    title="Highest temperatures"
)
fig.show()

In [92]:
dest = "Biarritz"
data_hotel = data_booking[data_booking["city"]== dest]
data_hotel.nlargest(20, ["review_score"])

,city,name,url,review_score,description,location,lat,lon
4,Biarritz,Hôtel du Palais Biarritz,https://www.booking.com/hotel/fr/hoteldupalais...,9.5,2 lits simples,"{'tourism': 'Hôtel du Palais', 'road': ""Avenue...",43.486435,-1.556328
20,Biarritz,Hôtel de La Plage,https://www.booking.com/hotel/fr/de-la-plage-b...,9.3,1 grand lit double,missing value,NaN,NaN
5,Biarritz,Hotel Saint Julien,https://www.booking.com/hotel/fr/saint-julien....,9.2,Types de lits : 1 lit double ou 2 lits simples,"{'tourism': 'Hôtel Saint-Julien', 'road': 'Ave...",43.479030,-1.562523
18,Biarritz,ALFRED HOTELS Port,https://www.booking.com/hotel/fr/georges-vi.fr...,9.2,1 grand lit double,missing value,NaN,NaN
23,Biarritz,Hotel Edouard VII,https://www.booking.com/hotel/fr/edouard-vii.f...,9.2,1 lit double,"{'amenity': 'Hôtel de ville', 'road': 'Avenue ...",43.483142,-1.558366
3,Biarritz,ALFRED HOTELS Les Halles,https://www.booking.com/hotel/fr/hotelanjou.fr...,9.1,1 grand lit double,missing value,NaN,NaN
8,Biarritz,Hôtel & Espace Bien,https://www.booking.com/hotel/fr/la-maison-du-...,9.1,Types de lits : 1 lit double ou 2 lits simples,missing value,NaN,NaN
14,Biarritz,Hotel de Silhouette,https://www.booking.com/hotel/fr/de-silhouette...,9.1,1 grand lit double,"{'tourism': 'Hôtel de Silhouette', 'road': 'Ru...",43.481062,-1.563382
9,Biarritz,Hôtel Le Café de Paris,https://www.booking.com/hotel/fr/le-cafe-de-pa...,9.0,"2 lits (1 lit simple, 1 grand lit double)",missing value,NaN,NaN
0,Biarritz,Hôtel Barnea,https://www.booking.com/hotel/fr/hotelargieder...,8.9,1 grand lit double,missing value,NaN,NaN


In [93]:
data_booking

,city,name,url,review_score,description,location,lat,lon
0,Biarritz,Hôtel Barnea,https://www.booking.com/hotel/fr/hotelargieder...,8.9,1 grand lit double,missing value,NaN,NaN
1,Biarritz,Le Talaia Hôtel & Spa Biarritz,https://www.booking.com/hotel/fr/radisson-blu-...,8.8,1 lit simple,"{'tourism': 'Le Talaia Hôtel & Spa Biarritz', ...",43.478210,-1.564422
2,Biarritz,Hôtel Kemaris,https://www.booking.com/hotel/fr/best-western-...,8.4,1 grand lit double,"{'tourism': 'Hôtel Best Western Kemaris', 'roa...",43.476518,-1.562496
3,Biarritz,ALFRED HOTELS Les Halles,https://www.booking.com/hotel/fr/hotelanjou.fr...,9.1,1 grand lit double,missing value,NaN,NaN
4,Biarritz,Hôtel du Palais Biarritz,https://www.booking.com/hotel/fr/hoteldupalais...,9.5,2 lits simples,"{'tourism': 'Hôtel du Palais', 'road': ""Avenue...",43.486435,-1.556328
...,...,...,...,...,...,...,...,...
122,Saintes Maries de la mer,Hôtel Le Neptune en Camargue,https://www.booking.com/hotel/fr/le-neptune-en...,8.4,Hébergement géré par un particulier,missing value,NaN,NaN
123,Saintes Maries de la mer,Le Sauvage,https://www.booking.com/hotel/fr/le-sauvage-le...,9.3,Hébergement géré par un particulier,missing value,NaN,NaN
124,Saintes Maries de la mer,De l'eau et des oiseaux !,https://www.booking.com/hotel/fr/de-l-eau-et-d...,8.8,Hébergement géré par un particulier,missing value,NaN,NaN
125,Saintes Maries de la mer,Hôtel Spa Mas des Rièges,https://www.booking.com/hotel/fr/mas-des-riege...,9.0,"2 lits (1 canapé-lit, 1 grand lit double)",missing value,NaN,NaN


In [94]:
data_gps

,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966
5,Paris,48.853495,2.348391
6,Amiens,49.894171,2.295695
7,Lille,50.636565,3.063528
8,Strasbourg,48.584614,7.750713
9,Chateau du Haut Koenigsbourg,48.249411,7.344320


In [100]:
cities = set(data_booking.city)
data_gps = data_gps.set_index(data_gps["city"])
for city in cities:
    data_city = data_booking[data_booking["city"] == city]
    fig = px.scatter_map(
        data_city,
        lat="lat", lon="lon",
        color="review_score",
        size="review_score", size_max=15,
        hover_name='city',
        hover_data={"description":True, "location":True},
        map_style="open-street-map",
        center={"lat":data_gps["lat"][city], "lon":data_gps["lon"][city]},
        zoom=11, height=720,
        title="Highest temperatures"
    )
    fig.show()      